# Model Combination dan Stacking Model

## 1. Import module dan library

In [1]:
import sys
sys.path.append('..')

# Import library yang dibutuhkan
import itertools # Digunakan untuk kombinasi model stacking
import pandas as pd # Digunakan untuk tabel hasil

from sklearn.model_selection import cross_validate, StratifiedKFold
# cross_validate → evaluasi multi-metrik
# StratifiedKFold → membagi data dengan menjaga distribusi label

from sklearn.ensemble import RandomForestClassifier,StackingClassifier
# StackingClassifier → metode ensemble stacking
# RandomForestClassifier → model berbasis tree

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from src import load_objek, simpan_objek

## 2. Alur Logika Code

### 2.1 Load data

In [13]:
data = load_objek ('../data/processed/data_vektor.pkl')

X_train, y_train = data['X_train'], data['y_train']  
# X_train → fitur (vektor TF-IDF)
# y_train → label

X_test, y_test = data['X_test_clean'],data['y_test']
# X_test = digunakan untuk test data


### 2.2 Setup base learners

#### 2.2.1 Parameter Default

In [3]:
base_models_default = {
    'NB': MultinomialNB(),

    'SVM': SVC(
        kernel='rbf',      # default sklearn
        probability=False  # default
    ),

    'RF': RandomForestClassifier(
        n_estimators=100,  # default
        random_state=42
    ),

    'LR': LogisticRegression(
        max_iter=100,      # default sklearn
    )
}

meta_learner = LogisticRegression()  
print(meta_learner.get_params())
# Meta learner (level-1) → menggabungkan hasil base models


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  
# Cross-validation 5 fold dengan distribusi label seimbang



{'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 100, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': None, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}


#### 2.2.2 Setelah di tuning

In [ ]:
base_models = {
    'NB': MultinomialNB(),

    # SVM: turunin agresivitas hoax
    'SVM': SVC(
        kernel='linear',
        probability=True,
        class_weight={0:1, 1:0.8}  # hoax diturunin
    ),

    # Random Forest: lebih stabil tanpa weight
    'RF': RandomForestClassifier(
        random_state=42,
        n_estimators=200,
        max_depth=None
    ),

    # Logistic Regression: dikontrol (ini penting banget)
    'LR': LogisticRegression(
        max_iter=1000,
        class_weight={0:1, 1:0.6},  # lebih ketat lagi
        C=0.5 
    )
}

meta_learner = LogisticRegression(
    C=0.5,                # 🔥 lebih kecil → kurangi overfitting
    penalty='l2',         # default tapi tetap ditulis (biar eksplisit)
    solver='lbfgs',       # stabil untuk dataset TF-IDF
    max_iter=1000,        # 🔥 biar pasti konvergen
    class_weight={0:1, 1:0.7},  # 🔥 kontrol bias ke hoax
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  
# Cross-validation 5 fold dengan distribusi label seimbang

### 2.3 Metrik evaluasi dan Eksperimen Kombinasi model

In [4]:
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_validate
from sklearn.ensemble import StackingClassifier
import itertools

# =========================
# SCORING (tetap kamu pakai)
# =========================
scoring_metrics = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='binary', zero_division=0),
    'recall': make_scorer(recall_score, average='binary', zero_division=0),
    'f1': make_scorer(f1_score, average='binary', zero_division=0)
}
# =========================
# EKSPERIMEN
# =========================
hasil_eksperimen_default = []

print("Memulai Eksperimen Kombinasi Stacking dengan Metrik Lengkap...\n")

for r in range(2, 5):
    kombinasi = list(itertools.combinations(base_models_default.items(), r))

    for pasang in kombinasi:

        # Nama kombinasi
        nama = " + ".join([m[0] for m in pasang])

        # Clone model biar tidak reuse object
        estimators = [(m[0], m[1].__class__(**m[1].get_params())) for m in pasang]

        # Stacking (FIX: cv pakai cv luar biar konsisten)
        stacking_clf = StackingClassifier(
            estimators=estimators,
            final_estimator=meta_learner.__class__(**meta_learner.get_params()),
            cv=cv,                 # FIX: pakai cv yang sama
            n_jobs=-1
        )

        # Cross validation
        scores = cross_validate(
            stacking_clf,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring_metrics,
            n_jobs=-1              
        )

        # Ambil rata-rata
        mean_acc = scores['test_accuracy'].mean()
        mean_prec = scores['test_precision'].mean()
        mean_rec = scores['test_recall'].mean()
        mean_f1 = scores['test_f1'].mean()

        # Simpan
        hasil_eksperimen_default.append({
            'Kombinasi': nama,
            'Accuracy': mean_acc,
            'Precision': mean_prec,
            'Recall': mean_rec,
            'F1-Score': mean_f1
        })

        # Print
        print(f"[{nama}]")
        print(f" -> Accuracy  : {mean_acc:.4f}")
        print(f" -> Precision : {mean_prec:.4f}")
        print(f" -> Recall    : {mean_rec:.4f}")
        print(f" -> F1-Score  : {mean_f1:.4f}\n")

Memulai Eksperimen Kombinasi Stacking dengan Metrik Lengkap...

[NB + SVM]
 -> Accuracy  : 0.8308
 -> Precision : 0.8447
 -> Recall    : 0.9655
 -> F1-Score  : 0.9010

[NB + RF]
 -> Accuracy  : 0.8268
 -> Precision : 0.8417
 -> Recall    : 0.9646
 -> F1-Score  : 0.8989

[NB + LR]
 -> Accuracy  : 0.8301
 -> Precision : 0.8383
 -> Recall    : 0.9753
 -> F1-Score  : 0.9016

[SVM + RF]
 -> Accuracy  : 0.8317
 -> Precision : 0.8478
 -> Recall    : 0.9622
 -> F1-Score  : 0.9013

[SVM + LR]
 -> Accuracy  : 0.8308
 -> Precision : 0.8457
 -> Recall    : 0.9638
 -> F1-Score  : 0.9009

[RF + LR]
 -> Accuracy  : 0.8317
 -> Precision : 0.8467
 -> Recall    : 0.9638
 -> F1-Score  : 0.9014

[NB + SVM + RF]
 -> Accuracy  : 0.8324
 -> Precision : 0.8481
 -> Recall    : 0.9626
 -> F1-Score  : 0.9017

[NB + SVM + LR]
 -> Accuracy  : 0.8314
 -> Precision : 0.8461
 -> Recall    : 0.9642
 -> F1-Score  : 0.9013

[NB + RF + LR]
 -> Accuracy  : 0.8311
 -> Precision : 0.8464
 -> Recall    : 0.9634
 -> F1-Score 

#### 2.3.2 Setelah tuning

In [54]:
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_validate
from sklearn.ensemble import StackingClassifier
import itertools

# =========================
# SCORING (tetap kamu pakai)
# =========================
scoring_metrics = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='binary', zero_division=0),
    'recall': make_scorer(recall_score, average='binary', zero_division=0),
    'f1': make_scorer(f1_score, average='binary', zero_division=0)
}
# =========================
# EKSPERIMEN
# =========================
hasil_eksperimen = []

print("Memulai Eksperimen Kombinasi Stacking dengan Metrik Lengkap...\n")

for r in range(2, 5):
    kombinasi = list(itertools.combinations(base_models.items(), r))

    for pasang in kombinasi:

        # Nama kombinasi
        nama = " + ".join([m[0] for m in pasang])

        # Clone model biar tidak reuse object
        estimators = [(m[0], m[1].__class__(**m[1].get_params())) for m in pasang]

        # Stacking (FIX: cv pakai cv luar biar konsisten)
        stacking_clf = StackingClassifier(
            estimators=estimators,
            final_estimator=meta_learner.__class__(**meta_learner.get_params()),
            cv=cv,                 # 🔥 FIX: pakai cv yang sama
            n_jobs=-1
        )

        # Cross validation
        scores = cross_validate(
            stacking_clf,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring_metrics,
            n_jobs=-1              # 🔥 tambah paralel di sini juga
        )

        # Ambil rata-rata
        mean_acc = scores['test_accuracy'].mean()
        mean_prec = scores['test_precision'].mean()
        mean_rec = scores['test_recall'].mean()
        mean_f1 = scores['test_f1'].mean()

        # Simpan
        hasil_eksperimen.append({
            'Kombinasi': nama,
            'Accuracy': mean_acc,
            'Precision': mean_prec,
            'Recall': mean_rec,
            'F1-Score': mean_f1
        })

        # Print
        print(f"[{nama}]")
        print(f" -> Accuracy  : {mean_acc:.4f}")
        print(f" -> Precision : {mean_prec:.4f}")
        print(f" -> Recall    : {mean_rec:.4f}")
        print(f" -> F1-Score  : {mean_f1:.4f}\n")

Memulai Eksperimen Kombinasi Stacking dengan Metrik Lengkap...

[NB + SVM]
 -> Accuracy  : 0.8347
 -> Precision : 0.8455
 -> Recall    : 0.9704
 -> F1-Score  : 0.9036

[NB + RF]
 -> Accuracy  : 0.8281
 -> Precision : 0.8434
 -> Recall    : 0.9638
 -> F1-Score  : 0.8995

[NB + LR]
 -> Accuracy  : 0.8268
 -> Precision : 0.8340
 -> Recall    : 0.9778
 -> F1-Score  : 0.9001

[SVM + RF]
 -> Accuracy  : 0.8377
 -> Precision : 0.8526
 -> Recall    : 0.9634
 -> F1-Score  : 0.9045

[SVM + LR]
 -> Accuracy  : 0.8360
 -> Precision : 0.8467
 -> Recall    : 0.9704
 -> F1-Score  : 0.9042

[RF + LR]
 -> Accuracy  : 0.8291
 -> Precision : 0.8446
 -> Recall    : 0.9634
 -> F1-Score  : 0.9000

[NB + SVM + RF]
 -> Accuracy  : 0.8360
 -> Precision : 0.8510
 -> Recall    : 0.9634
 -> F1-Score  : 0.9036

[NB + SVM + LR]
 -> Accuracy  : 0.8363
 -> Precision : 0.8470
 -> Recall    : 0.9704
 -> F1-Score  : 0.9044

[NB + RF + LR]
 -> Accuracy  : 0.8298
 -> Precision : 0.8452
 -> Recall    : 0.9634
 -> F1-Score 

### 2.4 Membuat tabel hasil

#### 2.4.1 Parameter Default

In [5]:

df_hasil_default = pd.DataFrame(hasil_eksperimen_default)  
# Mengubah list hasil menjadi DataFrame (tabel)

print("=== RINGKASAN HASIL EKSPERIMEN (DIURUTKAN BERDASARKAN F1-SCORE) ===")

print(
    df_hasil_default
    .sort_values(by='F1-Score', ascending=False)  # Urutkan berdasarkan F1-score tertinggi
    .to_string(index=False)                       # Tampilkan tanpa index
)


=== RINGKASAN HASIL EKSPERIMEN (DIURUTKAN BERDASARKAN F1-SCORE) ===
         Kombinasi  Accuracy  Precision   Recall  F1-Score
     SVM + RF + LR  0.832730   0.848881 0.961771  0.901746
     NB + SVM + RF  0.832401   0.848111 0.962593  0.901664
           NB + LR  0.830108   0.838323 0.975337  0.901595
NB + SVM + RF + LR  0.832073   0.847998 0.962182  0.901433
           RF + LR  0.831746   0.846744 0.963826  0.901424
          SVM + RF  0.831746   0.847752 0.962182  0.901281
     NB + SVM + LR  0.831417   0.846079 0.964236  0.901250
      NB + RF + LR  0.831090   0.846411 0.963415  0.901045
          NB + SVM  0.830760   0.844749 0.965470  0.901017
          SVM + LR  0.830761   0.845727 0.963826  0.900865
           NB + RF  0.826825   0.841666 0.964645  0.898873


#### 2.4.2 Hasil Tuning

In [56]:

df_hasil = pd.DataFrame(hasil_eksperimen)  
# Mengubah list hasil menjadi DataFrame (tabel)

print("=== RINGKASAN HASIL EKSPERIMEN (DIURUTKAN BERDASARKAN F1-SCORE) ===")

print(
    df_hasil
    .sort_values(by='F1-Score', ascending=False)  # Urutkan berdasarkan F1-score tertinggi
    .to_string(index=False)                       # Tampilkan tanpa index
)


=== RINGKASAN HASIL EKSPERIMEN (DIURUTKAN BERDASARKAN F1-SCORE) ===
         Kombinasi  Accuracy  Precision   Recall  F1-Score
NB + SVM + RF + LR  0.837977   0.852892 0.963412  0.904671
          SVM + RF  0.837650   0.852595 0.963413  0.904512
     NB + SVM + LR  0.836337   0.846956 0.970404  0.904412
          SVM + LR  0.836009   0.846651 0.970403  0.904239
     NB + SVM + RF  0.836009   0.850955 0.963413  0.903624
          NB + SVM  0.834698   0.845461 0.970406  0.903555
     SVM + RF + LR  0.834697   0.850006 0.963003  0.902907
      NB + RF + LR  0.829780   0.845169 0.963413  0.900343
           NB + LR  0.826830   0.833977 0.977807  0.900124
           RF + LR  0.829125   0.844571 0.963414  0.900010
           NB + RF  0.828140   0.843356 0.963825  0.899500


### 2.5 Latih model Final

#### 2.5.1 Parmeter Default

In [7]:
# ============================================================
# 5. Ambil Kombinasi Terbaik Otomatis (Lengkap dengan Metrik)
# ============================================================

# Urutkan berdasarkan F1-score tertinggi
df_sorted_default = df_hasil_default.sort_values(by='F1-Score', ascending=False)

# Ambil baris terbaik (bukan cuma nama)
best_row = df_sorted_default.iloc[0]

# Ambil informasi lengkap
best_combo = best_row['Kombinasi']
best_acc = best_row['Accuracy']
best_prec = best_row['Precision']
best_rec = best_row['Recall']
best_f1 = best_row['F1-Score']

# Tampilkan hasil lengkap
print("\n=== KOMBINASI MODEL TERBAIK ===")
print(f"Kombinasi : {best_combo}")
print(f"Accuracy  : {best_acc:.4f}")
print(f"Precision : {best_prec:.4f}")
print(f"Recall    : {best_rec:.4f}")
print(f"F1-Score  : {best_f1:.4f}")

# Pecah string kombinasi jadi list (contoh: "NB + SVM" → ['NB', 'SVM'])
best_keys = best_combo.split(" + ")

# Ambil model dari dictionary base_models
best_estimators = [(key, base_models_default[key]) for key in best_keys]

print("\nMelatih Model Final...")

final_model = StackingClassifier(
    estimators=best_estimators,
    final_estimator=LogisticRegression(),
    n_jobs=-1
)

# Training model menggunakan seluruh data training
final_model.fit(X_train, y_train)


=== KOMBINASI MODEL TERBAIK ===
Kombinasi : SVM + RF + LR
Accuracy  : 0.8327
Precision : 0.8489
Recall    : 0.9618
F1-Score  : 0.9017

Melatih Model Final...


,estimators,"[('SVM', ...), ('RF', ...), ...]"
,final_estimator,LogisticRegression()
,cv,None
,stack_method,'auto'
,n_jobs,-1
,passthrough,False
,verbose,0
,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'


#### 2.5.2 Hasil Tuning

In [57]:
# ============================================================
# 5. Ambil Kombinasi Terbaik Otomatis (Lengkap dengan Metrik)
# ============================================================

# Urutkan berdasarkan F1-score tertinggi
df_sorted = df_hasil.sort_values(by='F1-Score', ascending=False)

# Ambil baris terbaik (bukan cuma nama)
best_row = df_sorted.iloc[0]

# Ambil informasi lengkap
best_combo = best_row['Kombinasi']
best_acc = best_row['Accuracy']
best_prec = best_row['Precision']
best_rec = best_row['Recall']
best_f1 = best_row['F1-Score']

# Tampilkan hasil lengkap
print("\n=== KOMBINASI MODEL TERBAIK ===")
print(f"Kombinasi : {best_combo}")
print(f"Accuracy  : {best_acc:.4f}")
print(f"Precision : {best_prec:.4f}")
print(f"Recall    : {best_rec:.4f}")
print(f"F1-Score  : {best_f1:.4f}")

# Pecah string kombinasi jadi list (contoh: "NB + SVM" → ['NB', 'SVM'])
best_keys = best_combo.split(" + ")

# Ambil model dari dictionary base_models
best_estimators = [(key, base_models[key]) for key in best_keys]

print("\nMelatih Model Final...")

final_model = StackingClassifier(
    estimators=best_estimators,
    final_estimator=LogisticRegression(),
    n_jobs=-1
)

# Training model menggunakan seluruh data training
final_model.fit(X_train, y_train)


=== KOMBINASI MODEL TERBAIK ===
Kombinasi : NB + SVM + RF + LR
Accuracy  : 0.8380
Precision : 0.8529
Recall    : 0.9634
F1-Score  : 0.9047

Melatih Model Final...


,estimators,"[('NB', ...), ('SVM', ...), ...]"
,final_estimator,LogisticRegression()
,cv,None
,stack_method,'auto'
,n_jobs,-1
,passthrough,False
,verbose,0
,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [8]:
print("\n=== TOP 3 MODEL TERBAIK DEFAULT ===")
print(df_sorted_default.head(3).to_string(index=False))


=== TOP 3 MODEL TERBAIK DEFAULT ===
    Kombinasi  Accuracy  Precision   Recall  F1-Score
SVM + RF + LR  0.832730   0.848881 0.961771  0.901746
NB + SVM + RF  0.832401   0.848111 0.962593  0.901664
      NB + LR  0.830108   0.838323 0.975337  0.901595


In [ ]:
print("\n=== TOP 3 MODEL TERBAIK TUNING ===")
print(df_sorted.head(3).to_string(index=False))


=== TOP 3 MODEL TERBAIK ===
         Kombinasi  Accuracy  Precision   Recall  F1-Score
NB + SVM + RF + LR  0.837977   0.852892 0.963412  0.904671
          SVM + RF  0.837650   0.852595 0.963413  0.904512
     NB + SVM + LR  0.836337   0.846956 0.970404  0.904412


In [10]:
print("\n=== AMBIL MASING MASING MODEL TERBAIK ===")
from sklearn.model_selection import cross_validate
from sklearn.ensemble import StackingClassifier
import pandas as pd

# ============================================================
# 1. EVALUASI MODEL TUNGGAL
# ============================================================

hasil_single = []

print("=== EVALUASI MODEL TUNGGAL ===\n")

for nama, model in base_models_default.items():
    
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring_metrics,
        n_jobs=-1
    )
    
    hasil_single.append({
        'Model': nama,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1-Score': scores['test_f1'].mean()
    })
    
    print(f"[{nama}]")
    print(f"Accuracy  : {scores['test_accuracy'].mean():.4f}")
    print(f"Precision : {scores['test_precision'].mean():.4f}")
    print(f"Recall    : {scores['test_recall'].mean():.4f}")
    print(f"F1-Score  : {scores['test_f1'].mean():.4f}\n")

df_single = pd.DataFrame(hasil_single)


# ============================================================
# 2. AMBIL STACKING TERBAIK (TOP 1)
# ============================================================

print("\n=== EVALUASI STACKING TERBAIK ===\n")

# best_combo harus sudah ada dari hasil eksperimen sebelumnya
# contoh: "SVM + RF + LR"
best_keys = best_combo.split(" + ")

# Ambil model sesuai kombinasi terbaik
best_estimators = [(key, base_models_default[key]) for key in best_keys]

# Bangun stacking model terbaik
best_stacking = StackingClassifier(
    estimators=best_estimators,
    final_estimator=meta_learner,
    cv=cv,
    n_jobs=-1
)

# Evaluasi dengan cross validation
scores_stack = cross_validate(
    best_stacking,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring_metrics,
    n_jobs=-1
)

hasil_stacking = {
    'Model': f'Stacking ({best_combo})',
    'Accuracy': scores_stack['test_accuracy'].mean(),
    'Precision': scores_stack['test_precision'].mean(),
    'Recall': scores_stack['test_recall'].mean(),
    'F1-Score': scores_stack['test_f1'].mean()
}

df_stacking = pd.DataFrame([hasil_stacking])

print(f"[Stacking {best_combo}]")
print(f"Accuracy  : {scores_stack['test_accuracy'].mean():.4f}")
print(f"Precision : {scores_stack['test_precision'].mean():.4f}")
print(f"Recall    : {scores_stack['test_recall'].mean():.4f}")
print(f"F1-Score  : {scores_stack['test_f1'].mean():.4f}")


# ============================================================
# 3. GABUNGKAN HASIL
# ============================================================

df_perbandingan = pd.concat([df_single, df_stacking], ignore_index=True)

# Urutkan berdasarkan F1-Score tertinggi
df_perbandingan = df_perbandingan.sort_values(by='F1-Score', ascending=False)

print("\n=== PERBANDINGAN MODEL SINGLE VS STACKING ===")
print(df_perbandingan)


# ============================================================
# 4. (OPSIONAL) SIMPAN KE CSV BUAT SKRIPSI
# ============================================================

df_perbandingan.to_csv("perbandingan_model.csv", index=False)
print("\nFile disimpan: perbandingan_model.csv")


=== AMBIL MASING MASING MODEL TERBAIK ===
=== EVALUASI MODEL TUNGGAL ===

[NB]
Accuracy  : 0.8114
Precision : 0.8133
Recall    : 0.9914
F1-Score  : 0.8935

[SVM]
Accuracy  : 0.8173
Precision : 0.8189
Recall    : 0.9901
F1-Score  : 0.8964

[RF]
Accuracy  : 0.8147
Precision : 0.8243
Recall    : 0.9757
F1-Score  : 0.8936

[LR]
Accuracy  : 0.8163
Precision : 0.8152
Recall    : 0.9955
F1-Score  : 0.8964


=== EVALUASI STACKING TERBAIK ===

[Stacking SVM + RF + LR]
Accuracy  : 0.8327
Precision : 0.8489
Recall    : 0.9618
F1-Score  : 0.9017

=== PERBANDINGAN MODEL SINGLE VS STACKING ===
                      Model  Accuracy  Precision    Recall  F1-Score
4  Stacking (SVM + RF + LR)  0.832730   0.848881  0.961771  0.901746
1                       SVM  0.817317   0.818890  0.990136  0.896384
3                        LR  0.816333   0.815236  0.995478  0.896378
2                        RF  0.814689   0.824339  0.975740  0.893621
0                        NB  0.811412   0.813256  0.991368  0.89350

### 2.6 Simpan model

#### 2.6.1 Model Default

In [9]:
simpan_objek(final_model, '../models/stacking_final_default.pkl')  
# Menyimpan model agar bisa digunakan untuk prediksi di masa depan

print("Model final berhasil disimpan!")

Model final berhasil disimpan!


#### 2.6.2 Model Tuning

In [59]:
simpan_objek(final_model, '../models/stacking_final.pkl')  
# Menyimpan model agar bisa digunakan untuk prediksi di masa depan

print("Model final berhasil disimpan!")

Model final berhasil disimpan!


In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import StackingClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# =========================
# TOP 1 MODEL (FIX)
# =========================
estimators = [
    ('svm', SVC()),
    ('rf', RandomForestClassifier(random_state=42)),
    ('lr', LogisticRegression())
]

meta = LogisticRegression()

stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=meta,
    cv=cv,
    n_jobs=-1
)

# =========================
# PARAM GRID
# =========================
param_grid = {
    'svm__C': [0.1, 1, 10],
    'svm__kernel': ['linear'],
    'svm__probability': [True],

    'rf__n_estimators': [100, 200],
    'rf__max_depth': [None, 10],

    'lr__C': [0.1, 0.5, 1.0],
    'lr__max_iter': [500, 1000],

    'final_estimator__C': [0.1, 0.5, 1.0],
    'final_estimator__max_iter': [500, 1000]
}

# =========================
# GRID SEARCH
# =========================
grid = GridSearchCV(
    stacking,
    param_grid=param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Best Params :", grid.best_params_)
print("Best Score  :", grid.best_score_)

best_stacking_model = grid.best_estimator_

Fitting 5 folds for each of 432 candidates, totalling 2160 fits
Best Params : {'final_estimator__C': 0.5, 'final_estimator__max_iter': 1000, 'lr__C': 0.5, 'lr__max_iter': 500, 'rf__max_depth': 10, 'rf__n_estimators': 200, 'svm__C': 1, 'svm__kernel': 'linear', 'svm__probability': True}
Best Score  : 0.9054103961169689


In [15]:
simpan_objek(best_stacking_model, '../models/stacking_final_tuning.pkl')  
# Menyimpan model agar bisa digunakan untuk prediksi di masa depan

print("Model final berhasil disimpan!")

Model final berhasil disimpan!


In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# ============================================================
# 1. LATIH ULANG MODEL TERBAIK (WAJIB)
# ============================================================
best_stacking_model.fit(X_train, y_train)

# ============================================================
# 2. PREDIKSI KE DATA TEST
# ============================================================
y_pred = best_stacking_model.predict(X_test)

# ============================================================
# 3. HITUNG METRIK EVALUASI
# ============================================================
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='binary', zero_division=0)
rec = recall_score(y_test, y_pred, average='binary', zero_division=0)
f1 = f1_score(y_test, y_pred, average='binary', zero_division=0)

# ============================================================
# 4. TAMPILKAN HASIL
# ============================================================
print("=== HASIL PENGUJIAN STACKING (GRID SEARCH) ===")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-Score  : {f1:.4f}")

# ============================================================
# 5. CLASSIFICATION REPORT (OPSIONAL TAPI BAGUS)
# ============================================================
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, zero_division=0))

# ============================================================
# 6. CONFUSION MATRIX (OPSIONAL)
# ============================================================
cm = confusion_matrix(y_test, y_pred)
print("\n=== CONFUSION MATRIX ===")
print(cm)

=== HASIL PENGUJIAN STACKING (GRID SEARCH) ===
Accuracy  : 0.8336
Precision : 0.8523
Recall    : 0.9573
F1-Score  : 0.9018

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       0.67      0.34      0.45       154
           1       0.85      0.96      0.90       609

    accuracy                           0.83       763
   macro avg       0.76      0.65      0.68       763
weighted avg       0.82      0.83      0.81       763


=== CONFUSION MATRIX ===
[[ 53 101]
 [ 26 583]]
